In [1]:
import csv
import configparser
import os
import shutil
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

config = configparser.ConfigParser()
config.read("../config.ini")

['../config.ini']

In [2]:
base_dir = config["DEFAULT"]["folder_prefix"]

participants = [
    id
    for id in os.listdir(base_dir)
    if id != "Practice Data" and os.path.isdir(os.path.join(base_dir, id))
]

In [3]:
unprocessed = []

for participant_id in tqdm(participants, desc="Searching participants"):
    participant_path = os.path.join(base_dir, participant_id)
    for timepoint in os.listdir(participant_path):
        timepoint_path = os.path.join(participant_path, timepoint, "Images")
        converted_file = os.path.join(timepoint_path, ".converted")
        images_folder = os.path.join(timepoint_path, "images")

        # Check conditions for processing
        if (
            os.path.isdir(timepoint_path)
            and not os.path.exists(converted_file)
            and os.path.isdir(images_folder)
        ):
            image_files = os.listdir(images_folder)
            if image_files:
                unprocessed.append(
                    {
                        "filepath": timepoint_path,
                        "id": participant_id,
                        "timepoint": timepoint,
                        "num_images": len(image_files),
                    }
                )

Searching participants:   0%|          | 0/170 [00:00<?, ?it/s]

Searching participants: 100%|██████████| 170/170 [06:12<00:00,  2.19s/it]


In [4]:
csv_file = "unprocessed_folders.csv"
with open(csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file, fieldnames=["filepath", "id", "timepoint", "num_images"]
    )
    writer.writeheader()
    writer.writerows(unprocessed)

In [5]:
# Print the command that words with Square Eyes processor

folder_paths = " ".join([f'"{entry["filepath"]}"' for entry in unprocessed])
processing_command = f"python SquareEyes.py -f {folder_paths}"
print("Processing command:")
print(processing_command)

Processing command:
python SquareEyes.py -f "/mnt/z/Square_Eyes_DP20_Data/Participant_Data/Main Study/Community Sample/1019/Time_2/Images" "/mnt/z/Square_Eyes_DP20_Data/Participant_Data/Main Study/Community Sample/1061/Time_2/Images" "/mnt/z/Square_Eyes_DP20_Data/Participant_Data/Main Study/Community Sample/1063/Time_2/Images" "/mnt/z/Square_Eyes_DP20_Data/Participant_Data/Main Study/Community Sample/1107/Time_2/Images" "/mnt/z/Square_Eyes_DP20_Data/Participant_Data/Main Study/Community Sample/1113/Time_2/Images" "/mnt/z/Square_Eyes_DP20_Data/Participant_Data/Main Study/Community Sample/1116/Time_2/Images" "/mnt/z/Square_Eyes_DP20_Data/Participant_Data/Main Study/Community Sample/1121/Time_2/Images" "/mnt/z/Square_Eyes_DP20_Data/Participant_Data/Main Study/Community Sample/1123/Time_2/Images" "/mnt/z/Square_Eyes_DP20_Data/Participant_Data/Main Study/Community Sample/1142/Time_2/Images" "/mnt/z/Square_Eyes_DP20_Data/Participant_Data/Main Study/Community Sample/1148(1146)/Time_2/Images" 

In [7]:
def copy_file(src_file):
    relative_path = src_file.relative_to(src_base)
    dest_file = dst_dir / relative_path
    dest_file.parent.mkdir(parents=True, exist_ok=True)
    try:
        shutil.copy2(src_file, dest_file)
    except Exception as e:
        print(f"Failed to copy {src_file}: {e}")

In [ ]:
local_folder_paths = []
src_base = Path(config["DEFAULT"]["folder_prefix"])
dst_dir = Path(r"~/Projects/square_eyes_tmp").expanduser()

pbar = tqdm(unprocessed, desc="Copying folders", position=0, leave=True)

for entry in pbar:

    src_dir = Path(entry["filepath"])

    rel_path = src_dir.relative_to(src_base)
    new_path = dst_dir / rel_path
    local_folder_paths.append(new_path)

    all_files = []
    for root, _, files in os.walk(src_dir):
        for file in files:
            all_files.append(Path(root) / file)

    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = [executor.submit(copy_file, f) for f in all_files]
        for _ in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Copying files",
            position=1,
            leave=False,
        ):
            pass  # `tqdm` updates on each completion

Copying folders:   0%|          | 0/96 [02:41<?, ?it/s]


TqdmKeyError: "Unknown argument(s): {'keep': False}"

In [ ]:
# Print the command that words with Square Eyes processor

folder_paths = " ".join([f'"{str(x)}"' for x in local_folder_paths])
processing_command = f"python SquareEyes.py -f {folder_paths}"
print("Processing command:")
print(processing_command)

Processing command:
python SquareEyes.py -f "/home/tasanders/Projects/square_eyes_tmp/1012/Time_2/Images" "/home/tasanders/Projects/square_eyes_tmp/1017/Time_2/Images" "/home/tasanders/Projects/square_eyes_tmp/1020/Time_2/Images" "/home/tasanders/Projects/square_eyes_tmp/1025/Time_2/Images" "/home/tasanders/Projects/square_eyes_tmp/1030/Time_2/Images" "/home/tasanders/Projects/square_eyes_tmp/1065/Time_2/Images" "/home/tasanders/Projects/square_eyes_tmp/1068/Time_2/Images" "/home/tasanders/Projects/square_eyes_tmp/1080/Time_2/Images" "/home/tasanders/Projects/square_eyes_tmp/1089/Time_2/Images" "/home/tasanders/Projects/square_eyes_tmp/1105/Time_2/Images" "/home/tasanders/Projects/square_eyes_tmp/1128/Time_2/Images" "/home/tasanders/Projects/square_eyes_tmp/1186/Time_1/Images" "/home/tasanders/Projects/square_eyes_tmp/1201/Time_1/Images" "/home/tasanders/Projects/square_eyes_tmp/1412/Time_1/Images" "/home/tasanders/Projects/square_eyes_tmp/1441/Time_1/Images" "/home/tasanders/Projects/

In [51]:
for folder in tqdm(folder_paths.replace('"', "").split(" ")):
    folder = Path(folder)
    if not folder.exists():
        print(f"Folder {folder} does not exist")
        continue
    rel_dir = folder.relative_to(dst_dir)
    ntwk_dir = src_base / rel_dir

    for f in [
        ".converted",
        "Image Data Import.csv",
        "Square Eyes Detections.json",
        "SquareEyes Template.tdb",
    ]:
        src_file = folder / f
        dest_file = ntwk_dir / f
        if src_file.exists():
            shutil.copyfile(src_file, dest_file)
        else:
            print(f"File {src_file} does not exist")

 38%|███▊      | 11/29 [00:25<00:41,  2.28s/it]

File /home/tasanders/Projects/square_eyes_tmp/1186/Time_1/Images/.converted does not exist
File /home/tasanders/Projects/square_eyes_tmp/1186/Time_1/Images/Image Data Import.csv does not exist
File /home/tasanders/Projects/square_eyes_tmp/1186/Time_1/Images/Square Eyes Detections.json does not exist
File /home/tasanders/Projects/square_eyes_tmp/1186/Time_1/Images/SquareEyes Template.tdb does not exist


100%|██████████| 29/29 [01:09<00:00,  2.39s/it]
